# Correlation Stage -- Prep Work

This is groundwork for the next phase, not the full analysis. The previous notebook (`feature_importance_analysis.ipynb`) asked *does the model rely on this feature*. This one starts answering a different question: *what is the actual direction and strength of each feature's relationship to pain*, without a model in between.

What's here:
1. A reusable correlation function (Pearson + Spearman, FDR-corrected)
2. A run against both targets (pain rating, stimulus intensity) on `ds005285`
3. A data-quality check the function surfaced immediately -- worth fixing before the real correlation stage starts

This notebook is self-contained -- it loads `features_combined.csv` itself and does not depend on `feature_importance_analysis.ipynb`.


## 0. Imports + data


In [1]:
import numpy as np
import pandas as pd
from scipy.stats import pearsonr, spearmanr

pd.set_option('display.max_rows', 60)


In [2]:
df = pd.read_csv('features_combined.csv')
labeled = df.dropna(subset=['rating', 'laser_power']).copy()
print('trials with usable rating + laser_power label:', labeled.shape)
labeled['dataset'].value_counts()


trials with usable rating + laser_power label: (4618, 32)


dataset
ds005285    4618
Name: count, dtype: int64

In [3]:
drop_cols = ['dataset', 'subject', 'epoch', 'vertex_channel', 'sfreq', 'gamma_band_hz',
             'alpha_erd_pct', 'plv_FCz-CPz', 'rating', 'laser_power']
feature_cols = [c for c in labeled.columns if c not in drop_cols]

X = labeled[feature_cols].fillna(labeled[feature_cols].median(numeric_only=True))
y_pain     = labeled['rating'].astype(float)
y_stimulus = labeled['laser_power'].astype(float)

print(f'{len(feature_cols)} features, {len(X)} trials')


22 features, 4618 trials


## 1. Correlation function

Pearson catches linear relationships; Spearman catches any monotonic relationship (so if Spearman is notably stronger than Pearson for a feature, that's a flag the relationship is real but nonlinear). Benjamini-Hochberg FDR correction is applied so that testing ~20 features doesn't produce false positives by chance alone.


In [ ]:
def bh_correct(pvals):
    """Benjamini-Hochberg FDR correction. NaN p-values (e.g. constant features) pass through as NaN."""
    pvals = np.asarray(pvals, dtype=float)
    out = np.full_like(pvals, np.nan)
    valid = ~np.isnan(pvals)
    p = pvals[valid]
    n = len(p)
    order = np.argsort(p)
    ranked = p[order] * n / (np.arange(n) + 1)
    ranked = np.minimum.accumulate(ranked[::-1])[::-1]
    corrected = np.empty(n)
    corrected[order] = np.minimum(ranked, 1.0)
    out[valid] = corrected
    return out

def corr_table(X, y):
    rows = []
    for c in X.columns:
        if X[c].nunique() <= 1:          # constant feature -- correlation undefined
            rows.append({'feature': c, 'pearson_r': np.nan, 'pearson_p': np.nan,
                         'spearman_r': np.nan, 'spearman_p': np.nan, 'note': 'CONSTANT -- check extraction'})
            continue
        pr, pp = pearsonr(X[c], y)
        sr, sp = spearmanr(X[c], y)
        rows.append({'feature': c, 'pearson_r': pr, 'pearson_p': pp,
                     'spearman_r': sr, 'spearman_p': sp, 'note': ''})
    out = pd.DataFrame(rows).set_index('feature')
    out['pearson_p_fdr'] = bh_correct(out['pearson_p'].values)
    out['significant_fdr'] = out['pearson_p_fdr'] < 0.05
    return out.reindex(out['pearson_r'].abs().sort_values(ascending=False).index)


## 2. Run it: pain rating vs. stimulus intensity


In [5]:
pain_corr = corr_table(X, y_pain)
stim_corr = corr_table(X, y_stimulus)

constant_features = pain_corr[pain_corr['note'] == 'CONSTANT -- check extraction'].index.tolist()
print('CONSTANT features found (correlation undefined, likely an extraction bug):', constant_features)

pain_corr.drop(columns='note').head(10)


CONSTANT features found (correlation undefined, likely an extraction bug): ['psd_delta', 'psd_theta', 'psd_alpha']


,pearson_r,pearson_p,spearman_r,spearman_p,pearson_p_fdr,significant_fdr
feature,,,,,,
sample_entropy,-0.343474,5.053863e-128,-0.370614,2.519387e-150,9.602340e-127,True
hjorth_mobility,-0.309903,2.344354e-103,-0.331850,3.985250e-119,2.227136e-102,True
spectral_entropy,-0.308742,1.471922e-102,-0.319331,5.721208e-110,9.322175e-102,True
higuchi_fd,-0.281331,9.340847e-85,-0.286679,4.469963e-88,4.436902e-84,True
dfa,0.269791,7.719230e-78,0.272037,3.696521e-79,2.933307e-77,True
perm_entropy,-0.199827,8.339867e-43,-0.202099,9.171325e-44,2.640958e-42,True
p2_amp,0.174627,6.061555e-33,0.174938,4.671466e-33,1.645279e-32,True
psd_beta,0.173572,1.462620e-32,0.212747,2.057413e-48,3.473722e-32,True
beta_erd_pct,-0.159589,1.009101e-27,-0.205020,5.158983e-45,2.130324e-27,True


In [6]:
stim_corr.drop(columns='note').head(10)


,pearson_r,pearson_p,spearman_r,spearman_p,pearson_p_fdr,significant_fdr
feature,,,,,,
sample_entropy,-0.379704,2.719313e-158,-0.359499,6.301894e-141,4.869744e-157,True
higuchi_fd,-0.379394,5.126046e-158,-0.348563,4.861912e-132,4.869744e-157,True
hjorth_mobility,-0.362664,1.448982e-143,-0.339003,1.482174e-124,9.176889e-143,True
spectral_entropy,-0.359038,1.519016e-140,-0.337582,1.824659e-123,7.215328e-140,True
dfa,0.357843,1.473806e-139,0.322030,6.608292e-112,5.600462e-139,True
perm_entropy,-0.301745,7.928651e-98,-0.279590,1.085623e-83,2.510740e-97,True
psd_beta,0.301088,2.172083e-97,0.332892,6.577441e-120,5.895654e-97,True
p2_amp,0.224216,1.036842e-53,0.221717,1.570014e-52,2.462499e-53,True
p2_lat,-0.185036,7.581847e-37,-0.191631,1.918765e-39,1.600612e-36,True


## 3. Before the real correlation stage

- **Fix the extraction bug**: `psd_delta` / `psd_theta` / `psd_alpha` return 0 for every single trial right now -- they can't be evaluated at all until that's fixed.
- **Check direction against the literature**: alpha/beta features should come out negative (suppression), theta/gamma should come out positive, per Paper 1's framework -- worth a dedicated pass once the zeroed-out features are fixed.
- **Replicate outside ds005285**: apply this same `corr_table()` function to the blind-test datasets once their labels are matched, to see if these correlations hold up beyond one dataset.
- **Watch for redundant features**: a follow-up feature-to-feature correlation matrix (not done here yet) would show which features are just measuring the same thing twice (e.g. `gamma_power` vs. `psd_gamma`).
